In [1]:
from datasets import load_dataset
import numpy as np
import evaluate
from peft import get_peft_model, LoraConfig, TaskType, PeftModel, PeftConfig
from transformers import RobertaTokenizer, RobertaForSequenceClassification, Trainer, TrainingArguments

/home/lewy700/Documents/roberta-hypernet-custom/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:

# #* Continue finetuning using peft from checkpoint

# checkpoint_path = "./results_lora/checkpoint-1800" # "roberta-base"
# tokenizer = RobertaTokenizer.from_pretrained(checkpoint_path)

# peft_config = PeftConfig.from_pretrained(checkpoint_path)

# base_model = RobertaForSequenceClassification.from_pretrained(peft_config.base_model_name_or_path, num_labels=2)
# model = PeftModel.from_pretrained(base_model, checkpoint_path)

In [3]:

#* Init finetuning using peft 

# Load tokenizer and model
model_name = "roberta-base" # "roberta-base"
tokenizer = RobertaTokenizer.from_pretrained(model_name)
base_model = RobertaForSequenceClassification.from_pretrained(model_name, num_labels=2)

peft_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    inference_mode=False,
    r=8,
    lora_alpha=8
)
model = get_peft_model(base_model, peft_config)

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [4]:
# Total number of parameters
total_params = sum(p.numel() for p in base_model.parameters())

# Trainable parameters (typically only a subset with LoRA)
trainable_params = sum(p.numel() for p in base_model.parameters() if p.requires_grad)

print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

Total parameters: 125,534,212
Trainable parameters: 887,042


In [5]:
# Load CoLA dataset
dataset = load_dataset("glue", "cola")

In [6]:
# Tokenize
def tokenize_function(examples):
    return tokenizer(examples["sentence"], truncation=True, padding="max_length", max_length=512)

encoded_dataset = dataset.map(tokenize_function, batched=True)
encoded_dataset = encoded_dataset.rename_column("label", "labels")
encoded_dataset.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

In [7]:
metric = evaluate.load("glue", "cola")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

In [8]:
training_args = TrainingArguments(
    output_dir="./results_lora",
    eval_strategy="epoch",
    # eval_steps=500,
    save_strategy="epoch",
    # save_steps=500,
    learning_rate=4e-4,
    per_device_train_batch_size=16, #* maybe think about adding gradient accumulation
    gradient_accumulation_steps=2,
    per_device_eval_batch_size=32,
    num_train_epochs=80,
    logging_dir="./logs_lora",
    load_best_model_at_end=True,
    metric_for_best_model="matthews_correlation",
    dataloader_num_workers=4,
    warmup_ratio=0.06,
    lr_scheduler_type="linear",
    optim="adamw_torch"  
)

/home/lewy700/Documents/roberta-hypernet-custom/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:174: UserWarning: CUDA initialization: CUDA unknown error - this may be due to an incorrectly set up environment, e.g. changing env variable CUDA_VISIBLE_DEVICES after program start. Setting the available devices to be zero. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:109.)
  return torch._C._cuda_getDeviceCount() > 0


In [9]:
# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=encoded_dataset["train"],
    eval_dataset=encoded_dataset["validation"],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

/tmp/ipykernel_100267/599301337.py:2: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
No label_names provided for model class `PeftModelForSequenceClassification`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


In [ ]:
trainer.train()

/home/lewy700/Documents/roberta-hypernet-custom/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
